# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive template for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
This dataset is defined by a Croissant schema:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

It contains mass spectrometry protein abundance, sequence, and modification data from extracellular vesicles isolated from human mast cells.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and explore its description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Keywords: {', '.join(getattr(metadata, 'keywords', []))}")

## 2. Data Overview
Here we inspect the available RecordSets, their fields, and `@id` identifiers.

Note: In `mlcroissant`, RecordSets are the main tabular data entities. We will reference them *by their `@id`*, as required. Let us enumerate the record sets and their fields below.

In [ ]:
from mlcroissant.models.record_set import RecordSet

record_sets = list(metadata.record_sets)
print(f"Number of RecordSets: {len(record_sets)}\n")
for recset in record_sets:
    print(f"RecordSet: {recset.name}\n  @id: {recset.id}\n  Description: {getattr(recset, 'description', '')}")
    print("  Fields:")
    for field in recset.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()

## 3. Data Extraction
For analysis, let's load the main data RecordSet as a DataFrame. Use the `@id` from above for the record set of interest (e.g., the main proteins table).

The field `@id`s and column headers will mirror what's displayed above. We'll extract the first few records for exploration.

In [ ]:
# Choose the relevant record set(s) by @id.
# For this FAIR^2 dataset, the primary RecordSet is likely to have an @id like 'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034' (from 'distribution'), 
# but we confirm from the overview above (adjust if record set ids differ).
# Otherwise, we can just enumerate all.

record_set_ids = [rs.id for rs in record_sets]  # List of all available @id

dataframes = {}
for rec_id in record_set_ids:
    records = list(dataset.records(record_set=rec_id))
    if records:  # Only create a DataFrame if records exist
        df = pd.DataFrame(records)
        dataframes[rec_id] = df

# Display available DataFrame columns for each RecordSet
for rec_id, df in dataframes.items():
    print(f"RecordSet @id: {rec_id}")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head(3))
    print("-"*60)

# For this notebook, let's pick the first (main) RecordSet
main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]
print(f'Analyzing RecordSet: {main_record_set_id}')

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (e.g., 'Number of Peptides', 'Coverage (%)', or 'MW (kDa)') for analysis using its `@id`, and demonstrate filtering, normalization, and aggregation. Replace these field IDs with the real ones from the DataFrame.

In [ ]:
# Identify available numeric fields
print(f"Numeric candidates: {[col for col in df.columns if pd.api.types.is_numeric_dtype(df[col]) or df[col].dtype == 'O']}")

# Example usage:
# numeric_field_id = '<FIELD @id>' (replace with actual @id, e.g. 'MW (kDa)' if that's a column)
# group_field_id = '<FIELD @id>' (for categorical grouping, e.g. 'Sample Group')

# Let's attempt to pick a common field name (adjust as necessary):
possible_numeric_fields = [col for col in df.columns if 'MW' in col or 'Coverage' in col or 'Peptide' in col]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]  # e.g. 'MW (kDa)'
else:
    numeric_field = df.select_dtypes(include=['number']).columns[0]

# Ensure the field is numeric, try to convert if not
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = df[numeric_field].quantile(0.75)  # Filter top 25%
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} found.")
display(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Grouping by a likely categorical field (e.g., 'SampleID', 'Condition', etc.)
possible_group_fields = [col for col in df.columns if 'Group' in col or 'Sample' in col or 'Type' in col]
if possible_group_fields:
    group_field = possible_group_fields[0]
    grouped_df = (
        filtered_df.groupby(group_field)[numeric_field, f"{numeric_field}_normalized"].mean().sort_values(by=numeric_field, ascending=False)
    )
    print(f"Grouped data by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize the numeric field distribution and its relationship to a categorical field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Basic histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by group, if group_field exists
if 'group_field' in locals():
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xlabel(group_field)
    plt.ylabel(numeric_field)
    plt.show()

## 6. Conclusion
- We loaded the [FAIR² mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using `mlcroissant` and Python.
- Explored available RecordSets and fields by their `@id`s and names.
- Extracted tabular data and demonstrated numeric filtering, normalization, grouping, and visualization.

This notebook serves as a template; adapt field and RecordSet IDs and analysis to your workflow or hypotheses.